# OT Traffic Adaptation Strategy

This notebook demonstrates the methodology for adapting the NSL-KDD baseline intrusion detection model to live Operational Technology (OT) and SCADA environments in digitalized mining operations.

### Phase 1: Data Collection Pipeline

Our data acquisition strategy deploys mirroring taps at pilot mineral resource extraction plants. Network packets are captured on cloud-based AWS EC2 sniffer nodes. The raw PCAP stream is processed in real time by CICFlowMeter to output bidirectional flow CSV logs.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from src.data.ot_collector import OTTrafficCollector
from src.data.swat import SWaTLoader
from src.optimization.bwoa import BinaryWhaleOptimizer
from src.optimization.fitness import FeatureFitnessEvaluator

# Simulate OT traffic feature generation
collector = OTTrafficCollector(interface="eth1", output_dir="data/raw/")
mock_csv = collector.capture(duration_seconds=5)

raw_flow_df = collector.load_captured(mock_csv)
print("Raw CICFlowMeter Output Columns (First 5 Rows):")
print(raw_flow_df.head())

In [ ]:
# Show feature mapping from CICFlowMeter to NSL-KDD equivalents
mapped_ot_df = collector.align_features_to_nslkdd(raw_flow_df)
print(f"Aligned OT DataFrame shape: {mapped_ot_df.shape}")
print("Aligned OT DataFrame Features (First 5 Columns):")
print(mapped_ot_df.iloc[:, :5].head())

In [ ]:
# Demonstrate BWOA re-running on OT feature space
print("Running BWOA for features selection on aligned OT dataset...")
X_ot = mapped_ot_df.values
y_ot = np.random.randint(0, 2, size=(len(X_ot),))  # Mock targets

evaluator = FeatureFitnessEvaluator(alpha=0.88)
optimizer = BinaryWhaleOptimizer(
    n_agents=5,
    n_features=X_ot.shape[1],
    max_iter=2,
    fitness_fn=evaluator.calculate_fitness
)

best_ot_mask, ot_history = optimizer.optimize(X_ot, y_ot, X_ot, y_ot)
print(f"Selected {np.sum(best_ot_mask)} optimal features for the OT domain.")

In [ ]:
# Demonstrate transfer learning strategy (freeze CNN layers, retrain LSTM)
print("Loading NSL-KDD baseline model...")
try:
    base_model = tf.keras.models.load_model("models/cnn_lstm_nslkdd_baseline.keras")
except Exception:
    # Rebuild a dummy model if file is absent
    from src.models.cnn_lstm import build_cnn_lstm
    base_model = build_cnn_lstm(input_shape=(1, 41), n_classes=5)

print("\nOriginal Model Layers trainability status:")
for layer in base_model.layers:
    print(f"  Layer {layer.name} : Trainable = {layer.trainable}")

# Freeze Conv1D feature extractors
print("\nFreezing spatial extraction blocks (Conv1D) for domain transfer...")
for layer in base_model.layers:
    if "conv1d" in layer.name or "batch_normalization" in layer.name:
        layer.trainable = False

print("\nUpdated Model Layers trainability status:")
for layer in base_model.layers:
    print(f"  Layer {layer.name} : Trainable = {layer.trainable}")

### Domain Transfer Methodology Summary

This methodology forms the core of Phase 2. By freezing the spatial-pattern extraction layers (Conv1D) trained on broad baseline datasets, we preserve generic network anomaly features. Retraining only the recurrent (LSTM) and classification dense layers on target OT flow statistics minimizes computational cost, allowing local adaptations on resource-constrained deployment nodes.